# BP6 Gate 4 — Statistical Validation & Explainability
**Customer360 Navigator Enterprise Suite — GenAI Resolution Assistant**

## Why this gate looks different from BP1-5's own Gate 4

The Master Plan's generic Gate 4 row (output: "Bootstrap CI, calibration, confusion matrix, SHAP
sample"; exit criteria: "All checks numeric and reproducible; leakage re-confirmed"; compliance
touchpoint: "Independent-style validation record (SR 11-7 second-line analog); disparate-impact
check where applicable (ECOA/Reg B)") assumes a supervised classifier with predicted probabilities
to validate. **BP6 has none, at any gate.** This gate instead validates Gate 3's champion
**retrieval strategy** (`taxonomy_bucket_match`) — mapping each generic output onto its real BP6
analog:

- **"Bootstrap CI"** → a real percentile bootstrap 95% CI around the champion's real coverage
  rate, resampling the real, finite query-bucket population (n=9) with replacement. The result —
  `[0.0, 0.666667]` around a point estimate of `0.333333` — is honestly wide, because the real
  population is genuinely small (9 distinct common-taxonomy buckets), not because the method is
  flawed. Reported as-is, never narrowed by pretending a larger sample exists.
- **"calibration"** → no predicted probability exists to calibrate against a real outcome. This
  gate instead independently **reproduces** Gate 3's own recorded coverage number from scratch, in
  a fresh kernel session, off the real Gold-layer bucket counts directly — never by reading Gate
  3's config-recorded value. It reproduces bit-exact (`0.333333 == 0.333333`).
- **"confusion matrix"** → a real 2×2 cross-tab of bucket presence (BANKING77-side Y/N × CFPB-side
  Y/N) over every real bucket — 3 buckets present on both sides, 6 BANKING77-only, 0 CFPB-only.
  Disclosed explicitly as the structural analog, not a classifier confusion matrix: there is no
  predicted-vs-actual label pair here, only a real structural fact crossed against itself.
- **"SHAP sample"** → `taxonomy_bucket_match` is a fully transparent, deterministic lookup, never a
  black-box model, so real explainability means tracing each of the 3 real cross-corpus-hit
  buckets back to the exact real crosswalk config entries (`cfpb_product_candidates`, `confidence`,
  `rationale`, and the real BANKING77 category labels) that produced it — not approximating
  feature importance for a model that does not exist.
- **"leakage re-confirmed"** → BP6 has no supervised target to leak, so this checks the one real
  thing that could silently merge the two corpora: re-reads both real Gold layers' column names
  fresh in this gate's own kernel (never trusting Gate 1's/Gate 2's own claim) and confirms zero
  shared raw columns beyond the one deliberate, disclosed join field (`common_taxonomy_bucket`).
- **"disparate-impact check where applicable (ECOA/Reg B)"** → Master Plan Section 9's own
  applicability matrix (table 3, row 4) names only BP1, BP2, BP3, BP7, and "wherever any
  demographic-adjacent field could appear." BP6 is not listed, and this gate's own real column
  lists (Section 9 of the notebook) carry no protected-class or demographic-adjacent field —
  recorded as `NOT_APPLICABLE` with that real citation, not a guess.

## What this gate does

1. **Independent reproduction** (Section 5) — re-derives Gate 3's own champion coverage number
   from the real Gold-layer bucket counts, in a fresh kernel session, and asserts an exact match.
2. **Bootstrap CI** (Section 6) — `bootstrap_champion_coverage_ci`
   (`src/genai/bp6_retrieval_candidates.py`, extended this gate) — 1000 real resamples, seed 42.
3. **Bucket-availability crosstab** (Section 7) — `build_bucket_availability_crosstab`.
4. **Explainability trace** (Section 8) — `build_explainability_trace`, written to
   `gate4_explainability_trace.json`.
5. **Leakage re-confirmation** (Section 9) — `reconfirm_no_raw_column_leakage`.
6. **ECOA/Reg B applicability** (Section 10) — `NOT_APPLICABLE`, real Master Plan citation.
7. **Zero-GenAI-SDK-loaded re-check** (Section 11) — same pattern as Gates 1–3.
8. **Independent validation record** (Section 12) — BP6's real analog of the "SR 11-7 second-line
   record" compliance touchpoint, written to `gate4_independent_validation_record.json`.
9. **Config write** (Section 13) — nested `gate4_statistical_validation:` dict block.
10. **Structural integrity checks** (Section 14) — 13 named assertions, including that the
    crosstab's totals stay internally consistent with the independently reproduced numbers (not
    just matching each other by construction — both are derived from the same real bucket-count
    load, and this checks they didn't silently diverge across the notebook's own sections).

## No bugs caught this gate

Sandbox verification against the real staged Gold layers and real `taxonomy_mapping.yaml`
produced correct results on the first pass — coverage reproduced bit-exact, the crosstab and
explainability trace matched hand-derived expectations exactly (3 both-side buckets: ATM/card/
transfers, each with real confidence LOW/MEDIUM/HIGH), and the idempotency re-run produced
identical config values with no new gate markers.

## Prerequisite

BP6 Gate 3 must have already run for real (writes this config's Gate 3 block and the retrieval
strategy inventory entry this notebook reads). Already real-run confirmed as of this gate's
delivery. Per this project's standing execution-boundary rule, Claude never runs this notebook —
only the user does, in the `home_credit_env` Jupyter kernel.

## What this gate does NOT do

- **No ML training, no synthetic ground truth, no calibration curve on a predicted probability**
  — there is no probabilistic model here to calibrate.
- **No re-derivation of the taxonomy crosswalk** — `configs/taxonomy_mapping.yaml` is read, never
  rewritten.
- **No external call of any kind.** BP6's own GenAI call does not occur until Gate 5.
- **No financial-impact or illustrative-projection content, anywhere.**

## Real, disclosed design choices in this gate

- The bootstrap CI's width is reported honestly rather than hidden or narrowed — a real
  consequence of the real, small (n=9) bucket population, explicitly disclosed in both the
  notebook output and the saved validation record (`small_population_disclosure`).
- The "confusion matrix" and "SHAP sample" mappings are named explicitly as *structural analogs*,
  not literal reproductions — BP6 genuinely has neither a classifier's predicted/actual label pair
  nor a black-box model to explain, and this gate says so rather than manufacturing a fake one.
- The explainability trace's `real_cfpb_product_candidates` / `crosswalk_confidence` /
  `crosswalk_rationale` fields are read verbatim from `configs/taxonomy_mapping.yaml` — the same
  file BP1 Gate 2 built and every downstream BP already reuses — never re-authored or summarized.

Every real number in this gate is read directly from BP1's own real, already real-run-confirmed
Gold layers, `configs/taxonomy_mapping.yaml`, and Gate 3's own real recorded output. Nothing is
estimated, assumed, or synthesized.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP6 Gate 4 (Statistical Validation & Explainability)
notebook. Single consolidated code cell (platform convention). Idempotent.

BP6 has no classifier and no predicted-probability output at any gate, so the Master Plan's
generic Gate 4 row (output: "Bootstrap CI, calibration, confusion matrix, SHAP sample"; exit
criteria: "All checks numeric and reproducible; leakage re-confirmed"; compliance touchpoint:
"Independent-style validation record (SR 11-7 second-line analog); disparate-impact check where
applicable (ECOA/Reg B)") is mapped onto BP6's real shape: validating Gate 3's champion RETRIEVAL
STRATEGY (taxonomy_bucket_match), not a model. See src/genai/bp6_retrieval_candidates.py's own
Gate 4 section docstring for the full mapping rationale of each generic output onto its real BP6
analog. ECOA/Reg B: Master Plan Section 9's own applicability matrix (table 3, row 4) names only
BP1/BP2/BP3/BP7 - BP6 is not listed, so this gate records ECOA/Reg B as NOT_APPLICABLE with that
real citation, not a guess.
"""

import os, sys, json, warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (same resolver as every other notebook)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports + dependency/prerequisite check
# ============================================================
from taxonomy.taxonomy_mapper import load_mapping_config
from genai.bp6_retrieval_candidates import (
    load_taxonomy_linked_bucket_counts,
    reproduce_champion_coverage_independently,
    bootstrap_champion_coverage_ci,
    build_bucket_availability_crosstab,
    build_explainability_trace,
    reconfirm_no_raw_column_leakage,
)
from genai.bp6_evidence_prep import genai_sdk_modules_loaded

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp6_genai_resolution_assistant" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

BP6_CONFIG_PATH = CONFIGS_DIR / "bp6_genai_resolution_assistant.yaml"
TAXONOMY_MAPPING_PATH = CONFIGS_DIR / "taxonomy_mapping.yaml"
B77_GOLD_PATH = DATA_PROCESSED / "banking77_common_taxonomy_gold.parquet"
CFPB_GOLD_PATH = DATA_PROCESSED / "cfpb_common_taxonomy_gold.parquet"
GATE3_MARKER_TEXT = "Gate 3 (Retrieval Strategy Benchmark & Champion Selection)"

for p in (BP6_CONFIG_PATH, TAXONOMY_MAPPING_PATH, B77_GOLD_PATH, CFPB_GOLD_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} does not exist. Prerequisite: BP6 Gate 3 must have run for real at least once."
        )
_config_text_before = BP6_CONFIG_PATH.read_text(encoding="utf-8")
if GATE3_MARKER_TEXT not in _config_text_before:
    raise RuntimeError("BP6 Gate 3's own config block was not found - run BP6 Gate 3 first.")

GATE3_INVENTORY_PATH = ARTIFACTS_DIR / "gate3_retrieval_strategy_inventory_entry.json"
if not GATE3_INVENTORY_PATH.exists():
    raise FileNotFoundError(f"{GATE3_INVENTORY_PATH} does not exist - run BP6 Gate 3 first.")
with open(GATE3_INVENTORY_PATH, "r", encoding="utf-8") as f:
    gate3_inventory = json.load(f)
print("[OK] Prerequisites confirmed: BP6 Gate 3 config block + inventory entry both present.")

# ============================================================
# SECTION 4: Load real per-bucket counts (same real Gold layers Gate 3 used, never re-derived)
# ============================================================
mapping = load_mapping_config(TAXONOMY_MAPPING_PATH)
b77_bucket_counts, cfpb_bucket_counts = load_taxonomy_linked_bucket_counts(B77_GOLD_PATH, CFPB_GOLD_PATH)
print(
    f"[OK] Real bucket counts reloaded: {b77_bucket_counts.height} distinct buckets on the "
    f"BANKING77 side, {cfpb_bucket_counts.height} on the CFPB side."
)

# ============================================================
# SECTION 5: Independent reproduction of Gate 3's own champion coverage number - "All checks
# numeric and reproducible" exit criterion. Computed fresh in THIS notebook's own kernel session,
# never by reading Gate 3's config-recorded value.
# ============================================================
reproduced = reproduce_champion_coverage_independently(b77_bucket_counts, cfpb_bucket_counts)
gate3_recorded_coverage = gate3_inventory["champion_coverage"]
coverage_reproduces_exactly = reproduced["coverage"] == gate3_recorded_coverage
print(
    f"[OK] Independent reproduction of champion coverage: {reproduced['coverage']} "
    f"(Gate 3 recorded: {gate3_recorded_coverage}) - "
    f"{'MATCH' if coverage_reproduces_exactly else 'MISMATCH'}"
)

# ============================================================
# SECTION 6: Bootstrap CI around the champion's real coverage rate ("Bootstrap CI" output)
# ============================================================
bootstrap_result = bootstrap_champion_coverage_ci(
    b77_bucket_counts, cfpb_bucket_counts, n_bootstrap=1000, random_state=42
)
print(
    f"[OK] Bootstrap 95% CI on champion coverage: [{bootstrap_result['ci_lower_2p5']}, "
    f"{bootstrap_result['ci_upper_97p5']}] (point estimate {bootstrap_result['point_estimate_coverage']}, "
    f"n_bootstrap={bootstrap_result['n_bootstrap']}, real population n={bootstrap_result['n_query_buckets']})"
)

# ============================================================
# SECTION 7: Real bucket-availability crosstab ("confusion matrix" output, structural analog)
# ============================================================
crosstab = build_bucket_availability_crosstab(b77_bucket_counts, cfpb_bucket_counts)
print(
    f"[OK] Bucket-availability crosstab: {crosstab['both_sides_n']} both-side buckets, "
    f"{crosstab['banking77_only_n']} BANKING77-only, {crosstab['cfpb_only_n']} CFPB-only "
    f"(of {crosstab['total_real_buckets_in_crosstab']} real buckets total)."
)

# ============================================================
# SECTION 8: Explainability trace ("SHAP sample" output, structural analog - taxonomy_bucket_match
# is a fully transparent deterministic lookup, so its own real config IS its explanation)
# ============================================================
explainability_trace = build_explainability_trace(mapping, b77_bucket_counts, cfpb_bucket_counts)
print(f"[OK] Explainability trace built for {len(explainability_trace)} real cross-corpus-hit buckets.")
for entry in explainability_trace:
    print(
        f"       - {entry['bucket']}: confidence={entry['crosswalk_confidence']}, "
        f"{entry['n_real_banking77_categories_mapped_here']} real BANKING77 categories mapped, "
        f"real CFPB products={entry['real_cfpb_product_candidates']}"
    )

EXPLAINABILITY_TRACE_PATH = ARTIFACTS_DIR / "gate4_explainability_trace.json"
with open(EXPLAINABILITY_TRACE_PATH, "w", encoding="utf-8") as f:
    json.dump(explainability_trace, f, indent=2)
print(f"[SAVED] Explainability trace: {EXPLAINABILITY_TRACE_PATH}")

# ============================================================
# SECTION 9: Leakage re-confirmation ("leakage re-confirmed" exit criterion) - re-reads both real
# Gold layers' column names fresh in this kernel, never trusting Gate 1's/Gate 2's own claim.
# ============================================================
leakage_check = reconfirm_no_raw_column_leakage(B77_GOLD_PATH, CFPB_GOLD_PATH)
print(
    f"[OK] Leakage re-confirmation: no_undisclosed_leakage={leakage_check['no_undisclosed_leakage']} "
    f"(shared columns beyond the deliberate join field: {leakage_check['shared_columns_excluding_deliberate_join_field']})"
)

# ============================================================
# SECTION 10: ECOA/Reg B disparate-impact-check applicability. Master Plan Section 9's own
# applicability matrix (table 3, row 4) names BP1/BP2/BP3/BP7 - BP6 is not listed, so this is
# recorded as NOT_APPLICABLE with that real citation, never a guess.
# ============================================================
ecoa_reg_b_status = "NOT_APPLICABLE"
ecoa_reg_b_rationale = (
    "Master Plan Section 9's own ECOA/Regulation B applicability row (table 3, row 4) names only "
    "BP1, BP2, BP3, BP7, and 'wherever any demographic-adjacent field could appear'. BP6's real "
    "inputs (BP1's real BANKING77 Gold text/category columns, and the real common_taxonomy_bucket "
    "column on both sides) carry no protected-class or demographic-adjacent field - re-confirmed "
    "structurally by Section 9 above (real column lists carry no such field). No disparate-impact "
    "testing is applicable to BP6's retrieval-strategy validation."
)
print(f"[OK] ECOA/Reg B: {ecoa_reg_b_status}")

# ============================================================
# SECTION 11: Zero-GenAI-SDK-loaded re-check (same pattern as Gates 1-3)
# ============================================================
loaded_sdks = genai_sdk_modules_loaded()
print(f"[OK] GenAI SDK modules loaded in this run: {loaded_sdks or 'NONE'}")

# ============================================================
# SECTION 12: Write the independent-style validation record - BP6's real analog of Gate 4's own
# "Independent-style validation record (SR 11-7 second-line analog)" compliance touchpoint.
# ============================================================
generated_at_utc = datetime.now(timezone.utc).isoformat()

validation_record = {
    "bp_id": "bp6",
    "gate": 4,
    "compliance_touchpoint": "Independent-style validation record (SR 11-7 second-line analog - "
    "BP6 has no classifier, so this record validates a retrieval strategy, not a model); "
    "disparate-impact check where applicable (ECOA/Reg B)",
    "champion_strategy_under_validation": gate3_inventory["champion_strategy"],
    "gate3_recorded_coverage": gate3_recorded_coverage,
    "gate4_independently_reproduced_coverage": reproduced["coverage"],
    "coverage_reproduces_exactly": coverage_reproduces_exactly,
    "bootstrap_ci": {k: v for k, v in bootstrap_result.items() if k != "small_population_disclosure"},
    "bucket_availability_crosstab": {
        k: v for k, v in crosstab.items()
        if k not in ("both_sides_buckets", "banking77_only_buckets", "cfpb_only_buckets")
    },
    "explainability_trace_path": EXPLAINABILITY_TRACE_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "n_explainability_trace_entries": len(explainability_trace),
    "leakage_reconfirmed": leakage_check["no_undisclosed_leakage"],
    "leakage_check_detail": leakage_check,
    "ecoa_reg_b_status": ecoa_reg_b_status,
    "ecoa_reg_b_rationale": ecoa_reg_b_rationale,
    "genai_sdk_modules_loaded_this_run": loaded_sdks,
    "generated_at_utc": generated_at_utc,
}
VALIDATION_RECORD_PATH = ARTIFACTS_DIR / "gate4_independent_validation_record.json"
with open(VALIDATION_RECORD_PATH, "w", encoding="utf-8") as f:
    json.dump(validation_record, f, indent=2)
print(f"[SAVED] Independent validation record: {VALIDATION_RECORD_PATH}")

# ============================================================
# SECTION 13: Write the Gate 4 config block. Nested dict block (gate4_statistical_validation),
# matching this whole project's established convention (nested blocks start at Gate 3).
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

gate4_marker = (
    "# --- Gate 4 (Statistical Validation & Explainability) results (appended, idempotent overwrite) ---"
)
gate4_block_lines = [
    "gate4_statistical_validation:",
    f'  champion_strategy_under_validation: "{gate3_inventory["champion_strategy"]}"',
    f"  gate3_recorded_coverage: {gate3_recorded_coverage}",
    f"  gate4_independently_reproduced_coverage: {reproduced['coverage']}",
    f"  coverage_reproduces_exactly: {coverage_reproduces_exactly}",
    f"  bootstrap_ci_lower_2p5: {bootstrap_result['ci_lower_2p5']}",
    f"  bootstrap_ci_upper_97p5: {bootstrap_result['ci_upper_97p5']}",
    f"  bootstrap_n: {bootstrap_result['n_bootstrap']}",
    f"  crosstab_both_sides_n: {crosstab['both_sides_n']}",
    f"  crosstab_banking77_only_n: {crosstab['banking77_only_n']}",
    f"  crosstab_cfpb_only_n: {crosstab['cfpb_only_n']}",
    f'  explainability_trace_path: "{EXPLAINABILITY_TRACE_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f"  leakage_reconfirmed: {leakage_check['no_undisclosed_leakage']}",
    f'  ecoa_reg_b_status: "{ecoa_reg_b_status}"',
    f'  independent_validation_record_path: "{VALIDATION_RECORD_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f"  genai_sdk_modules_loaded_this_run: {loaded_sdks}",
    f'  generated_at_utc: "{generated_at_utc}"',
]
write_gate_block(BP6_CONFIG_PATH, gate4_marker, gate4_block_lines)
print(f"[SAVED] Gate 4 config block written to {BP6_CONFIG_PATH}")

# ============================================================
# SECTION 14: Structural integrity checks
# ============================================================
_config_text_after = BP6_CONFIG_PATH.read_text(encoding="utf-8")
_front_matter_and_priors_preserved = all(
    marker in _config_text_after
    for marker in (
        'bp_id: "bp6"',
        GATE3_MARKER_TEXT,
        "pii_screen_rows_scanned:",
        "gate3_retrieval_benchmark:",
        "random_state: 42",
    )
)

_checks: list[tuple[str, bool]] = [
    ("coverage_reproduces_exactly_matching_gate3", coverage_reproduces_exactly),
    ("bootstrap_ci_bounds_valid_and_ordered", 0.0 <= bootstrap_result["ci_lower_2p5"] <= bootstrap_result["point_estimate_coverage"] <= bootstrap_result["ci_upper_97p5"] <= 1.0),
    ("bootstrap_point_estimate_matches_reproduced_coverage", bootstrap_result["point_estimate_coverage"] == reproduced["coverage"]),
    ("crosstab_totals_consistent_with_reproduced_query_bucket_count", crosstab["total_real_buckets_in_crosstab"] == reproduced["n_query_buckets"]),
    ("crosstab_both_sides_n_matches_reproduced_hit_count", crosstab["both_sides_n"] == reproduced["n_buckets_with_cross_corpus_hit"]),
    ("explainability_trace_covers_every_both_sides_bucket", len(explainability_trace) == crosstab["both_sides_n"]),
    ("explainability_trace_written", EXPLAINABILITY_TRACE_PATH.exists()),
    ("leakage_reconfirmed_true", leakage_check["no_undisclosed_leakage"]),
    ("ecoa_reg_b_not_applicable_disclosed", ecoa_reg_b_status == "NOT_APPLICABLE"),
    ("zero_genai_sdk_modules_loaded", loaded_sdks == []),
    ("validation_record_written", VALIDATION_RECORD_PATH.exists()),
    ("config_gate4_block_written", gate4_marker in _config_text_after),
    ("config_front_matter_and_prior_gate_blocks_preserved", _front_matter_and_priors_preserved),
]

_failed = [name for name, ok in _checks if not ok]
for name, ok in _checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert not _failed, f"BP6 Gate 4 structural integrity checks failed: {_failed}"

print(
    "\n[ALL CHECKS PASSED] BP6 Gate 4 (Statistical Validation & Explainability) complete. "
    f"Champion strategy {gate3_inventory['champion_strategy']}'s coverage independently reproduces "
    f"at {reproduced['coverage']} (95% bootstrap CI [{bootstrap_result['ci_lower_2p5']}, "
    f"{bootstrap_result['ci_upper_97p5']}]). Leakage re-confirmed: "
    f"{leakage_check['no_undisclosed_leakage']}. ECOA/Reg B: {ecoa_reg_b_status}. This gate is "
    "prep-only - BP6's own retrieval/generation work begins at Gate 5."
)
